In [ ]:
import pandas as pd

In [ ]:
bench_data = pd.read_json('roseau.json')

bench_data.head()

In [ ]:
all_results = [key for key in bench_data.iloc[0]['counts'].keys()]
all_results.sort()

GROUND_TRUTH_KEY = 'GroundTruth'
tools_analyzed = [result for result in all_results if result != GROUND_TRUTH_KEY]

full_data = pd.DataFrame(columns=['bench'])
full_data['bench'] = bench_data['name']

GENERAL_BREAKING_SUFFIX = 'b'
BINARY_BREAKING_SUFFIX = 'bb'
SOURCE_BREAKING_SUFFIX = 'sb'
for result in all_results:
    full_data[f'{result}_{GENERAL_BREAKING_SUFFIX}'] = bench_data['counts'].apply(lambda x: x[result]['isSourceBreaking'] | x[result]['isBinaryBreaking'])
    full_data[f'{result}_{BINARY_BREAKING_SUFFIX}'] = bench_data['counts'].apply(lambda x: x[result]['isBinaryBreaking'])
    full_data[f'{result}_{SOURCE_BREAKING_SUFFIX}'] = bench_data['counts'].apply(lambda x: x[result]['isSourceBreaking'])

full_data.head()

In [ ]:
def compute_metrics_for_breaking_type(data, suffix, tools):
    metrics_data = pd.DataFrame(columns=['tool', 'precision', 'recall', 'f1'])

    for tool in tools:
        tp = data[(data[f'{tool}_{suffix}'] == True) & (data[f'{GROUND_TRUTH_KEY}_{suffix}'] == True)].count().iloc[0]
        fp = data[(data[f'{tool}_{suffix}'] == True) & (data[f'{GROUND_TRUTH_KEY}_{suffix}'] == False)].count().iloc[0]
        fn = data[(data[f'{tool}_{suffix}'] == False) & (data[f'{GROUND_TRUTH_KEY}_{suffix}'] == True)].count().iloc[0]

        precision = tp / (tp + fp)
        recall = tp / (tp + fn)
        f1 = 2 * (precision * recall) / (precision + recall)

        metrics_data = pd.concat(
            [
                metrics_data,
                pd.DataFrame.from_records([{
                    'tool': tool,
                    'precision': precision,
                    'recall': recall,
                    'f1': f1
                }])
            ],
            ignore_index=True
        )

    return metrics_data


In [ ]:
general_breaking_metrics_data = compute_metrics_for_breaking_type(full_data, GENERAL_BREAKING_SUFFIX, tools_analyzed)

general_breaking_metrics_data

In [ ]:
binary_breaking_metrics_data = compute_metrics_for_breaking_type(full_data, BINARY_BREAKING_SUFFIX, tools_analyzed)

binary_breaking_metrics_data

In [ ]:
source_breaking_metrics_data = compute_metrics_for_breaking_type(full_data, SOURCE_BREAKING_SUFFIX, tools_analyzed)

source_breaking_metrics_data

In [ ]:
roseau_general_fp = full_data[(full_data['RoseauJar_b'] == True) & (full_data[f'{GROUND_TRUTH_KEY}_b'] == False)][['bench', 'GroundTruth_b', 'RoseauJar_b']]
roseau_general_fn = full_data[(full_data[f'RoseauJar_b'] == False) & (full_data[f'{GROUND_TRUTH_KEY}_b'] == True)][['bench', 'GroundTruth_b', 'RoseauJar_b']]

roseau_binary_fp = full_data[(full_data['RoseauJar_bb'] == True) & (full_data[f'{GROUND_TRUTH_KEY}_bb'] == False)][['bench', 'GroundTruth_bb', 'RoseauJar_bb']]
roseau_binary_fn = full_data[(full_data[f'RoseauJar_bb'] == False) & (full_data[f'{GROUND_TRUTH_KEY}_bb'] == True)][['bench', 'GroundTruth_bb', 'RoseauJar_bb']]

roseau_source_fp = full_data[(full_data['RoseauJar_sb'] == True) & (full_data[f'{GROUND_TRUTH_KEY}_sb'] == False)][['bench', 'GroundTruth_sb', 'RoseauJar_sb']]
roseau_source_fn = full_data[(full_data[f'RoseauJar_sb'] == False) & (full_data[f'{GROUND_TRUTH_KEY}_sb'] == True)][['bench', 'GroundTruth_sb', 'RoseauJar_sb']]

roseau_general_fp
# roseau_general_fn
# roseau_binary_fp
# roseau_binary_fn
# roseau_source_fp
# roseau_source_fn